# 🚀 VerifAI-ML Training on Google Colab

This notebook trains the AI image detection model using Google Colab's free GPU resources.

## 📋 Training Plan:
1. Setup environment and mount Google Drive
2. Download datasets (reduced size for Colab)
3. Prepare and augment data
4. Train YOLOv8 model
5. Save trained model to Google Drive

**Expected time**: 5-6 hours
**Storage needed**: ~50GB (handled by Colab)

## 🔧 Step 1: Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

# Install dependencies
print("Installing dependencies...")
!pip install torch torchvision ultralytics numpy pandas Pillow opencv-python scikit-learn matplotlib plotly seaborn PyYAML tqdm datasets -q

print("✅ Dependencies installed!")

In [ ]:
# Mount Google Drive for saving the model
from google.colab import drive
drive.mount('/content/drive')

# Create project directory
!mkdir -p /content/VerifAI-ML
%cd /content/VerifAI-ML

print("✅ Google Drive mounted and project directory created!")

## 📥 Step 2: Download Datasets

In [ ]:
import os
from datasets import load_dataset
from tqdm import tqdm
from PIL import Image

# Create directories
!mkdir -p dataset/real dataset/ai_generated

print("📥 Downloading AI-generated images (5000 images)...")
# Scaling up the dataset to give the model more examples to learn from
dataset = load_dataset("ash12321/sdxl-generated-10k", split="train[:5000]")

for i, item in enumerate(tqdm(dataset, desc="AI Images")):
    image = item['image']
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image.save(f"dataset/ai_generated/ai_sdxl_{i}.jpg", "JPEG", quality=95)

print("✅ AI images downloaded!")

In [ ]:
print("📥 Downloading real images (5000 images)...")
# Using Food101: A natively high-resolution dataset to replace the 32x32 CIFAR-10 images
real_dataset = load_dataset("food101", split="train[:5000]")

for i, item in enumerate(tqdm(real_dataset, desc="Real Images")):
    image = item['image']
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Resize to match AI images but starting from a HIGH resolution source
    image = image.resize((512, 512), Image.Resampling.LANCZOS)
    image.save(f"dataset/real/real_{i}.jpg", "JPEG", quality=95)

print("✅ Real images downloaded!")
print(f"Dataset size: {len(os.listdir('dataset/ai_generated'))} AI + {len(os.listdir('dataset/real'))} Real images")

## 🔄 Step 3: Upload Training Scripts

In [ ]:
# Create the training scripts directly in Colab
import os

# Create src directory structure
!mkdir -p src/data src/models src/utils

# Create __init__.py files
with open('src/__init__.py', 'w') as f:
    f.write('__version__ = "1.0.0"\n')
with open('src/data/__init__.py', 'w') as f:
    f.write('')
with open('src/models/__init__.py', 'w') as f:
    f.write('')
with open('src/utils/__init__.py', 'w') as f:
    f.write('')

print("✅ Directory structure created!")

In [ ]:
# Create data preparation script
data_prep_script = '''
import os
import shutil
import random
from io import BytesIO
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DataPreparator:
    def __init__(self, dataset_path="dataset", output_path="yolo_dataset"):
        self.dataset_path = Path(dataset_path)
        self.output_path = Path(output_path)
        self.classes = ["real", "ai_generated"]
        
    def create_yolo_structure(self):
        splits = ["train", "val", "test"]
        for split in splits:
            split_path = self.output_path / split
            split_path.mkdir(parents=True, exist_ok=True)
            for class_name in self.classes:
                class_path = split_path / class_name
                class_path.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created YOLOv8 directory structure in {self.output_path}")
    
    def aggressive_augmentation(self, image_path, save_dir, base_name):
        try:
            img = Image.open(image_path)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            augmented_images = []
            
            # Original
            augmented_images.append((img.copy(), f"{base_name}_original.jpg"))
            
            # Horizontal flip
            if random.random() > 0.3:
                flipped = img.transpose(Image.FLIP_LEFT_RIGHT)
                augmented_images.append((flipped, f"{base_name}_flip.jpg"))
            
            # Simulated Internet Compression (Crucial for AI detection)
            if random.random() > 0.5:
                buffer = BytesIO()
                # Compress heavily then read back
                img.save(buffer, "JPEG", quality=random.randint(30, 60))
                buffer.seek(0)
                compressed = Image.open(buffer)
                augmented_images.append((compressed, f"{base_name}_compressed.jpg"))
            
            # Slight Blur
            if random.random() > 0.6:
                blurred = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
                augmented_images.append((blurred, f"{base_name}_blur.jpg"))
            
            # Save augmented images
            for aug_img, filename in augmented_images:
                aug_img.save(save_dir / filename, "JPEG", quality=95)
            
            return len(augmented_images)
            
        except Exception as e:
            logger.error(f"Error augmenting {image_path}: {e}")
            return 0
    
    def prepare_dataset(self, train_ratio=0.8, val_ratio=0.1):
        self.create_yolo_structure()
        
        for class_name in self.classes:
            class_path = self.dataset_path / class_name
            images = list(class_path.glob("*.jpg")) + list(class_path.glob("*.png"))
            
            random.shuffle(images)
            
            n_total = len(images)
            n_train = int(n_total * train_ratio)
            n_val = int(n_total * val_ratio)
            
            train_images = images[:n_train]
            val_images = images[n_train:n_train + n_val]
            test_images = images[n_train + n_val:]
            
            # Copy and augment training data
            for img_path in tqdm(train_images, desc=f"Processing {class_name} train"):
                base_name = img_path.stem
                self.aggressive_augmentation(img_path, self.output_path / "train" / class_name, base_name)
            
            # Copy validation and test data (no augmentation)
            for img_path in tqdm(val_images, desc=f"Processing {class_name} val"):
                shutil.copy2(img_path, self.output_path / "val" / class_name / img_path.name)
            
            for img_path in tqdm(test_images, desc=f"Processing {class_name} test"):
                shutil.copy2(img_path, self.output_path / "test" / class_name / img_path.name)
        
        self.print_dataset_summary()
    
    def print_dataset_summary(self):
        print("\\n=== Dataset Summary ===")
        for split in ["train", "val", "test"]:
            total = 0
            for class_name in self.classes:
                class_path = self.output_path / split / class_name
                count = len(list(class_path.glob("*.jpg")) + list(class_path.glob("*.png")))
                print(f"{split}/{class_name}: {count} images")
                total += count
            print(f"{split} total: {total} images")
        print("========================\\n")

if __name__ == "__main__":
    from tqdm import tqdm
    preparator = DataPreparator()
    preparator.prepare_dataset()
'''

with open('src/data/data_preparation.py', 'w') as f:
    f.write(data_prep_script)

print("✅ Data preparation script created!")

In [ ]:
# Create training pipeline script
training_script = '''
import torch
import torchvision
from ultralytics import YOLO
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import logging
from tqdm import tqdm
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class TrainingPipeline:
    # Upgraded to the Large YOLO classification model
    def __init__(self, dataset_path="yolo_dataset", model_name="yolov8l-cls.pt"):
        self.dataset_path = Path(dataset_path)
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"Using device: {self.device}")
    
    # Dropped batch_size to 8 to fit larger model in Colab VRAM, increased epochs
    def train_model(self, epochs=100, img_size=224, batch_size=8):
        model = YOLO(self.model_name)
        
        training_args = {
            'data': str(self.dataset_path),
            'epochs': epochs,
            'imgsz': img_size,
            'batch': batch_size,
            'device': str(self.device),
            'patience': 15,  # Increased patience for longer training
            'lr0': 0.001,
            'optimizer': 'AdamW',
            'augment': True,
            'cache': True,
            'save_period': 5,
            'plots': True,
            'verbose': True
        }
        
        logger.info("Starting training...")
        results = model.train(**training_args)
        
        model.save('best.pt')
        logger.info("Training completed! Model saved as best.pt")
        
        return results, model
    
    def evaluate_model(self, model):
        logger.info("Evaluating model...")
        results = model.val(data=str(self.dataset_path))
        
        print(f"\\n=== Model Performance ===")
        print(f"mAP@0.5: {results.box.map:.4f}" if hasattr(results, 'box') else "Accuracy metrics saved to runs/ folder")
        print("========================\\n")
        
        return results

if __name__ == "__main__":
    trainer = TrainingPipeline()
    # Allowing up to 100 epochs, but early stopping will likely catch it sooner
    results, model = trainer.train_model(epochs=100) 
    evaluation = trainer.evaluate_model(model)
'''

with open('src/models/training_pipeline.py', 'w') as f:
    f.write(training_script)

print("✅ Training pipeline script created!")

## 🔄 Step 4: Prepare Dataset

In [ ]:
print("🔄 Preparing and augmenting dataset...")
!python src/data/data_preparation.py

print("✅ Dataset preparation completed!")

## 🏋️ Step 5: Train the Model

In [ ]:
print("🏋️ Starting model training...")
print("This will take 20-30 minutes. Feel free to grab a coffee! ☕")

!python src/models/training_pipeline.py

print("🎉 Training completed!")

## 💾 Step 6: Save Model to Google Drive

In [ ]:
# Copy trained model to Google Drive
!mkdir -p /content/drive/MyDrive/VerifAI-ML
!cp best.pt /content/drive/MyDrive/VerifAI-ML/
!cp -r runs /content/drive/MyDrive/VerifAI-ML/

print("✅ Model and training results saved to Google Drive!")
print("📁 Location: /content/drive/MyDrive/VerifAI-ML/")

# Show model file size
!ls -lh best.pt

print("\n🎉 Training complete! You can now:")
print("1. Download the model from your Google Drive")
print("2. Use it with the Streamlit app")
print("3. Share it with others")

## 📊 Training Results Summary

After training completes, you'll have:
- **best.pt**: Trained YOLOv8 model (~25MB)
- **runs/**: Training curves and metrics
- **Model performance**: Expected 70-85% accuracy with reduced dataset

## 🔄 Next Steps

1. **Download the model** from Google Drive to your local machine
2. **Update the local app** to use the trained model
3. **Test the model** with your Streamlit app

## 🐛 Troubleshooting

- **Out of memory**: Reduce batch size in training script
- **Slow training**: Make sure GPU is enabled (Runtime > Change runtime type > GPU)
- **Connection issues**: Re-run the download cells

---

**🎉 Congratulations! You've trained an AI image detection model!**